# DCF Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Turn DCF model outputs into a compact valuation page with enterprise value, equity bridge, cash-flow forecast, and sensitivities.

**Prerequisites:** `04_statement_modeling/models/dcf_valuation.ipynb`.

**What you'll learn:**

- Provide DCF forecast and valuation inputs.
- Render `reporting.dcf_tearsheet` inline.
- Use WACC and terminal-growth sensitivities as reviewer context.

Project unlevered free cash flow, run a Gordon-growth DCF, and render a
`dcf_tearsheet` — the EV→equity bridge, the UFCF projection, a WACC /
terminal-growth sensitivity tornado, and a forecast summary.


In [ ]:
import datetime as dt
import json

from finstack_quant import reporting
from finstack_quant.core.money import Money
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import dcf_sensitivity, evaluate_dcf

qs = ["2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2", "2026Q3", "2026Q4"]
b = ModelBuilder("acme-dcf")
b.periods("2025Q1..2026Q4", "2025Q2")
b.with_meta("currency", json.dumps("USD"))
b.value_money("revenue", [(period, Money(value, "USD")) for period, value in zip(qs, [100, 105, 110, 116, 122, 128, 134, 140], strict=True)])
b.compute("ebitda", "revenue * 0.27")
b.compute("ufcf", "ebitda * 0.6")
b.compute("net_income", "ebitda * 0.5")
spec = b.build()
result = Evaluator().evaluate(spec)

TV = json.dumps({"type": "gordon_growth", "growth_rate": 0.025})
NET_DEBT = 51.0
WACC = 0.10

dcf = evaluate_dcf(spec, WACC, TV, "ufcf", net_debt_override=NET_DEBT)
print("Enterprise value:", round(dcf.enterprise_value.amount, 1), " Equity value:", round(dcf.equity_value.amount, 1))


## WACC & terminal-growth sensitivity

`statements_analytics.dcf_sensitivity` ranks the headline DCF assumptions by
enterprise-value impact. The statement model is evaluated once and each shocked
point re-runs only the DCF, so this is materially cheaper than re-running
`evaluate_dcf` end to end per point.

Conventions to note:

- `wacc_sensitivity_bump` (default `0.01`) shocks **both** WACC and the terminal
  growth rate by ±100 bp; `exit_multiple_bump` applies instead when the terminal
  value is an exit multiple.
- `downside`/`upside` are **deltas versus the baseline enterprise value**, and
  they are keyed to the *parameter* leg: `downside` is the down-shock of the
  assumption (lower WACC ⇒ higher EV). Entries arrive sorted by descending
  absolute swing.
- The WACC down-shock is clamped so that `wacc - g` stays above
  `wacc_denominator_epsilon` and the Gordon-growth denominator cannot blow up.
  The `*_clamped` flags report whether the clamp bound.

Because a flat `net_debt_override` is used here, EV deltas and equity-value
deltas are identical, so these rows read directly as equity sensitivity.

In [ ]:
sens = dcf_sensitivity(spec, WACC, TV, "ufcf", net_debt_override=NET_DEBT)
sensitivity = sens.entries

baseline_ev = sens.baseline_enterprise_value
print("Baseline EV:", round(baseline_ev.amount, 1), baseline_ev.currency.code)
print("WACC down leg:", round(sens.wacc_down, 4), "clamped:", sens.wacc_down_clamped)
print("Terminal growth up leg:", round(sens.terminal_growth_up, 4), "clamped:", sens.terminal_growth_up_clamped)
for s in sensitivity:
    print(f"{s.parameter_id:18s} down={s.downside:+.1f}  up={s.upside:+.1f}")


## Valuation tear sheet

In [ ]:
presentation_report = reporting.dcf_tearsheet(
    dcf,
    results=result,
    sensitivity=sensitivity,
    title="Acme Corp — DCF",
    generated=dt.date(2026, 6, 22),
)
# Replace the helper's static numeric-mode caption with provenance guidance.
presentation_report.meta_lines=["Source calculations determine arithmetic; display formatting only"]
presentation_report


## Saving a standalone HTML file

```python
ts = reporting.dcf_tearsheet(dcf, results=result, sensitivity=sensitivity, generated=dt.date(2026, 6, 22))
ts.save("dcf_tearsheet.html")
```


## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
